In [1]:
import pandas as pd
import numpy as np
import re

In [2]:
def clean_generic_name(name):
    if pd.isna(name) or name == 'nan':
        return name
    
    # 1. 統一標點符號：全型斜線與逗號轉半型，移除常見括號內容
    name = name.replace('／', '/').replace('，', ',')
    name = re.sub(r'[\(\uff08].*?[\)\uff09]', '', name)
    
    # 2. 強力移除劑量與單位 (處理 100,000, 1%, 80 mg/ml, 2.4 MIU 等)
    # 匹配數字、逗號、點、百分比，後接常見容量單位
    name = re.sub(r'[\d\.,\s]+(g|mg|ml|units|miu|％|%)\b', ' ', name, flags=re.IGNORECASE)
    # 針對單獨殘留的 /ml 或 %
    name = re.sub(r'/\s*ml|%', '', name, flags=re.IGNORECASE)
    
    # 3. 移除特定劑型關鍵字與無義詞
    forms = ['inj', 'tab', 'cap', 'susp', 'iv', 'oral', 'for', 'solution', 'hydrate', 'sodium', 'benzathine']
    pattern_forms = r'\b(' + '|'.join(forms) + r')\b'
    name = re.sub(pattern_forms, '', name, flags=re.IGNORECASE)

    # 4. 特定藥名結構處理 (依據您的要求)
    # Amphotericin B liposome -> Amphotericin B/liposome
    name = re.sub(r'Amphotericin B liposome', 'Amphotericin B/liposome', name, flags=re.IGNORECASE)

    # 5. 【核心強化】消除斜線 (/) 前後的任何空格
    # \s* 代表 0 到多個空白字元
    name = re.sub(r'\s*/\s*', '/', name)

    
    # 5. 清理殘留符號：移除多餘空格、末尾點號與斜線
    name = re.sub(r'\s+', ' ', name) # 多空格轉單空格
    name = name.strip(' ./,')       # 移除前後的空格、點、斜線、逗號
    
    # 6. 字典對照 (處理特殊轉換)
    mapping = {
        'R +I': 'ifampin/Isoniazid',
        'R 300 +I 150': 'ifampin/Isoniazid',
        'Penicillin G .': 'Penicillin G',
        'Penicillin G benzathine': 'Penicillin G',
        'Baktar': 'Sulfamethoxazole/Trimethoprim',
        'Clindamycin 1 ％' : 'Clindamycin',
        'Penicillin 5 MU' : 'Penicillin',
        'Rifampin, Isoniazid and Ethambutol' : 'Rifampin/Isoniazid/Ethambutol',
        'Minocycline injection' : 'Minocycline',
        'Amoxicillin/Clavulanate': 'Amoxicillin/Clavulanic acid'
    }
    
    # 如果完全符合字典 key，或是處理後變成 key 的樣子就轉換
    return mapping.get(name, name)

In [3]:
df2024 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2024\2024Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})
df2025 = pd.read_csv(r'C:\Users\482525\Desktop\敗血症資料\2025\2025Sepsis00114.csv', encoding='big5', dtype={'VERIFYDATE': str, 'STARTTIME': str, 'ENDTIME': str, 0: str, 21: str, 22: str})

df2024['VERIFYDATE'] = pd.to_datetime(df2024['VERIFYDATE'], format='%Y%m%d', errors='coerce')
df2025['VERIFYDATE'] = pd.to_datetime(df2025['VERIFYDATE'], format='%Y%m%d', errors='coerce')
df2024['STARTTIME'] = pd.to_datetime(df2024['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2024['ENDTIME'] = pd.to_datetime(df2024['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['STARTTIME'] = pd.to_datetime(df2025['STARTTIME'], format='%Y%m%d%H%M%S', errors='coerce')
df2025['ENDTIME'] = pd.to_datetime(df2025['ENDTIME'], format='%Y%m%d%H%M%S', errors='coerce')

table14 = pd.concat([df2024, df2025], ignore_index=True)

In [4]:
table14 = table14.dropna(how='all')
table14 = table14[table14['ACCOUNTNO'].notna()]
table14 = table14.loc[:, ~table14.columns.str.contains('^Unnamed')]

In [5]:
len(table14), len(table14['ACCOUNTNO'].unique())

(86059, 27968)

In [6]:
table14['VERIFYDATE'] = pd.to_datetime(table14['VERIFYDATE'], format='%Y%m%d%H%M', errors='coerce')
table14['STARTTIME'] = pd.to_datetime(table14['STARTTIME'], format='%Y%m%d%H%M', errors='coerce')
table14['ENDTIME'] = pd.to_datetime(table14['ENDTIME'], format='%Y%m%d%H%M', errors='coerce')

In [7]:
# table14[table14['ACCOUNTNO'] == 'I11300000002']

In [8]:
table14['GENERICNAME_Clear'] = table14['GENERICNAME'].apply(clean_generic_name)

In [9]:
table14['GENERICNAME_Clear'].unique()

array(['Flomoxef', 'Tenofovir alafenamide', 'Cefixime',
       'Amoxicillin/Clavulanic acid', 'Metronidazole', 'Cefazolin',
       'Ceftriaxone', 'Cefoperazone/sulbactam', 'Gentamicin',
       'Cefadroxil', 'Oseltamivir', 'Baloxavir marboxil', 'Clindamycin',
       'Piperacillin/Tazobactam', 'Cefepime', 'Azithromycin',
       'Levofloxacin', 'Acyclovir', 'Cefuroxime', 'Ciprofloxacin',
       'Doxycycline', 'Peramivir', 'Vancomycin', 'Amoxicillin',
       'Valaciclovir', 'Nystatin', 'Ceftazidime', 'Penicillin',
       'Doripenem', 'Ertapenem', 'Cefotaxime', 'Cephalexin', 'Oxacillin',
       'Sulfamethoxazole/Trimethoprim', 'Ampicillin', 'Itraconazole',
       'Fluconazole', 'Amikacin', 'Moxifloxacin', 'Clarithromycin',
       'Anidulafungin', 'Ceftazidime/Avibactam', 'Fenticonazole',
       'Fosfomycin', 'Pipemidic acid', 'Cefoxitin',
       'Rifampin/Isoniazid/Ethambutol', 'Pyrazinamide', 'Teicoplanin',
       'Meropenem', 'Minocycline', 'ifampin/Isoniazid', 'Ceftizoxime',
       'Famc

In [10]:
table14['GENERICNAME_Clear'][table14['GENERICNAME_Clear'].isna() == True]

Series([], Name: GENERICNAME_Clear, dtype: object)

In [11]:
table14[['ACCOUNTNO', 'GENERICNAME_Clear']]

,ACCOUNTNO,GENERICNAME_Clear
0,I11300000002,Flomoxef
1,I11300000002,Flomoxef
2,I11300000002,Flomoxef
3,I11300000002,Tenofovir alafenamide
4,I11300000002,Tenofovir alafenamide
...,...,...
86055,I11400060720,Piperacillin/Tazobactam
86056,I11400060731,Flomoxef
86057,I11400060731,Flomoxef
86058,I11400060731,Flomoxef


In [12]:
infection_cols = [
    "INFECTIONSITE1",
    "INFECTIONSITE2",
    "INFECTIONSITE3",
    "INFECTIONSITE4",
    "INFECTIONSITE5",
    "INFECTIONSITE9"
]

for col in infection_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [13]:
table14['INFECTIONSITE1']

0        1
1        0
2        1
3        0
4        0
        ..
86055    1
86056    1
86057    0
86058    1
86059    1
Name: INFECTIONSITE1, Length: 86059, dtype: int32

In [14]:
BACTERIA_cols = [
    "BACTERIA1",
    "BACTERIA2",
    "BACTERIA3",
    "BACTERIA4",
    "BACTERIA5",
    "BACTERIA9"
]

for col in BACTERIA_cols:
    table14[col] = (table14[col] == "Y").astype(int)

In [15]:
table14['BACTERIA1']

0        0
1        0
2        1
3        0
4        0
        ..
86055    0
86056    0
86057    0
86058    0
86059    0
Name: BACTERIA1, Length: 86059, dtype: int32

In [16]:
table14['AUTIBIOTICRANK'].unique()

array(['A32', 'A11', 'A21', 'A42', 'A22', 'A52', 'C42'], dtype=object)

In [17]:
rank_mapping = {
    'A11': 1,
    'A21': 2,
    'A22': 2,
    'A32': 2,
    'A42': 3,
    'A52': 3,
    'C42': 3
}

In [18]:
table14['AUTIBIOTIC_GROUP'] = table14['AUTIBIOTICRANK'].map(rank_mapping)

In [19]:
last_index = table14.groupby('ACCOUNTNO')['VERIFYDATE'].min().reset_index()
table14_last = pd.merge(table14, last_index, on=['ACCOUNTNO', 'VERIFYDATE'])

In [20]:
table14_last

,DIVISIONNO,ACCOUNTNO,IPDACCOUNTNO,SOURCEFROM,SEX,AGE,VERIFYDATE,STARTTIME,ENDTIME,ORDERNO,...,REASON12,REASON13,REASON14,REASON15,REASON16,REASON17,REASON99,OTHERREASON,GENERICNAME_Clear,AUTIBIOTIC_GROUP
0,001,I11300000002,I11300000176,E,M,65.0,2024-01-01,2024-01-01 01:04:06,NaT,PF29,...,N,N,N,N,N,N,N,NaN,Flomoxef,2
1,001,I11300000002,I11300000176,I,M,65.0,2024-01-01,2024-01-01 01:04:06,2024-01-03 14:00:04,PF29,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Flomoxef,2
2,001,I11300000003,NaN,E,F,48.0,2024-01-01,2024-01-01 00:05:06,2024-01-06 00:05:06,OA66,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Amoxicillin/Clavulanic acid,2
3,001,I11300000008,NaN,E,F,19.0,2024-01-01,2024-01-01 04:00:08,2024-01-06 04:00:08,OC94,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Cefixime,2
4,001,I11300000008,NaN,E,F,19.0,2024-01-01,2024-01-01 04:00:08,2024-01-06 04:00:08,OM22,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Metronidazole,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
51220,001,I11400060701,I11500000064,I,M,68.0,2026-01-02,2026-01-02 19:00:01,2026-01-08 11:03:02,PF29,...,N,N,N,N,N,N,N,NaN,Flomoxef,2
51221,001,I11400060720,I11500000056,E,M,71.0,2025-12-31,2025-12-31 22:00:01,NaT,PT27,...,N,N,N,N,N,N,N,NaN,Piperacillin/Tazobactam,2
51222,001,I11400060720,I11500000056,I,M,71.0,2025-12-31,2025-12-31 22:00:01,2026-01-02 13:04:09,PT27,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Piperacillin/Tazobactam,2
51223,001,I11400060731,I11500000059,E,M,62.0,2025-12-31,2025-12-31 23:00:09,NaT,PF29,...,N,N,N,N,N,N,N,NaN,Flomoxef,2


In [21]:
# len(table14_last['GENERICNAME_Clear'].unique())

In [22]:
table14_last = table14_last[['ACCOUNTNO', 'GENERICNAME_Clear']].drop_duplicates()

In [23]:
table14_last.head(10)

,ACCOUNTNO,GENERICNAME_Clear
0,I11300000002,Flomoxef
2,I11300000003,Amoxicillin/Clavulanic acid
3,I11300000008,Cefixime
4,I11300000008,Metronidazole
5,I11300000011,Cefixime
6,I11300000014,Cefixime
7,I11300000014,Metronidazole
8,I11300000015,Cefazolin
10,I11300000015,Ceftriaxone
11,I11300000022,Amoxicillin/Clavulanic acid


In [24]:
table14_last['GENERICNAME_Clear'].unique(), len(table14_last['GENERICNAME_Clear'].unique())

(array(['Flomoxef', 'Amoxicillin/Clavulanic acid', 'Cefixime',
        'Metronidazole', 'Cefazolin', 'Ceftriaxone', 'Oseltamivir',
        'Baloxavir marboxil', 'Azithromycin', 'Levofloxacin', 'Acyclovir',
        'Doxycycline', 'Peramivir', 'Cefoperazone/sulbactam', 'Vancomycin',
        'Piperacillin/Tazobactam', 'Amoxicillin', 'Ciprofloxacin',
        'Valaciclovir', 'Cefepime', 'Doripenem', 'Cephalexin',
        'Cefadroxil', 'Clindamycin', 'Ampicillin', 'Gentamicin',
        'Cefuroxime', 'Ceftazidime', 'Cefotaxime', 'Moxifloxacin',
        'Ertapenem', 'Pipemidic acid', 'Cefoxitin', 'Fosfomycin',
        'Fenticonazole', 'Oxacillin', 'Meropenem', 'Amikacin',
        'Tenofovir alafenamide', 'Clarithromycin',
        'tenofovir/emtricitabine/bictegravir',
        'Sulfamethoxazole/Trimethoprim',
        'abacavir/lamivudine/dolutegravir', 'Rifampin', 'Isoniazid',
        'Ampicillin/Sulbactam', 'Nystatin',
        'Rifampin/Isoniazid/Ethambutol', 'Famciclovir', 'Minocycline',
    

In [25]:
abx14 = (table14_last.assign(value=1)
                     .pivot_table(index=['ACCOUNTNO'],columns='GENERICNAME_Clear',values='value',fill_value=0))

In [26]:
# add = table14_last.groupby('ACCOUNTNO')['AUTIBIOTIC_GROUP'].max().reset_index()
# abx14 = abx14.merge(add, on='ACCOUNTNO', how='left')

In [27]:
# infection_cols = ['INFECTIONSITE1', 'INFECTIONSITE2', 'INFECTIONSITE3', 
#                   'INFECTIONSITE4', 'INFECTIONSITE5', 'INFECTIONSITE9']

# for col in infection_cols:
#     table14[col] = table14[col].map({'Y': 1, 'N': 0}).fillna(0).astype(int)

# binary OTHERINFECTIONSITE 
table14['OTHERINFECTIONSITE_flag'] = (
    table14['OTHERINFECTIONSITE'].fillna('').str.strip().ne('').astype(int)
)

infects_summary = table14.groupby('ACCOUNTNO').agg({
    **{col: 'max' for col in infection_cols}, 
    'OTHERINFECTIONSITE_flag': 'max'
}).reset_index()


abx14 = abx14.merge(infects_summary, on='ACCOUNTNO', how='left').fillna(0)

In [28]:
abx14.columns

Index(['ACCOUNTNO', 'Acyclovir', 'Amikacin', 'Amoxicillin',
       'Amoxicillin/Clavulanic acid', 'Amphotericin B',
       'Amphotericin B/liposome', 'Ampicillin', 'Ampicillin/Sulbactam',
       'Azithromycin', 'Baloxavir marboxil', 'Cefadroxil', 'Cefazolin',
       'Cefepime', 'Cefixime', 'Cefoperazone/sulbactam', 'Cefotaxime',
       'Cefoxitin', 'Ceftazidime', 'Ceftazidime/Avibactam', 'Ceftizoxime',
       'Ceftriaxone', 'Cefuroxime', 'Cephalexin', 'Ciprofloxacin',
       'Clarithromycin', 'Clindamycin', 'Dicloxacillin', 'Doripenem',
       'Doxycycline', 'Ertapenem', 'Erythromycin', 'Ethambutol', 'Famciclovir',
       'Fenticonazole', 'Flomoxef', 'Fluconazole', 'Flucytosine', 'Fosfomycin',
       'Ganciclovir', 'Gentamicin', 'Griseofulvin', 'Imipenem/Cilastatin',
       'Isoniazid', 'Itraconazole', 'Lamivudine/Dolutegravir', 'Levofloxacin',
       'Linezolid', 'Meropenem', 'Metronidazole', 'Micafungin', 'Minocycline',
       'Moxifloxacin', 'Nemonoxacin', 'Nystatin', 'Oseltamivir',

In [29]:
abx14

,ACCOUNTNO,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Amphotericin B,Amphotericin B/liposome,Ampicillin,Ampicillin/Sulbactam,Azithromycin,...,tenofovir/emtricitabine,tenofovir/emtricitabine/bictegravir,tenofovir/emtricitabine/rilpivirine,INFECTIONSITE1,INFECTIONSITE2,INFECTIONSITE3,INFECTIONSITE4,INFECTIONSITE5,INFECTIONSITE9,OTHERINFECTIONSITE_flag
0,I11300000002,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
1,I11300000003,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
2,I11300000008,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
3,I11300000011,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
4,I11300000014,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27963,I11400060686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0
27964,I11400060687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
27965,I11400060701,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0,1,0,0,0,0,0
27966,I11400060720,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1,0,0,0,0,0,0


In [30]:
table14_clear = abx14.reset_index(drop=True)

In [31]:
table14_clear.to_csv('table14_clear.csv', index=False)

In [32]:
table14_clear = table14_clear.drop(columns=['ACCOUNTNO']) # 'AUTIBIOTIC_GROUP'

In [33]:
start_index = table14_clear.columns.get_loc('Acyclovir')
end_index = table14_clear.columns.get_loc('tenofovir/emtricitabine/rilpivirine')

abx_cols = table14_clear.columns[start_index:end_index+1]
col_sum = table14_clear[abx_cols].sum()

final_cols = col_sum[col_sum >= 0].index.tolist() # 抗生素
base_cols = [c for c in table14_clear.columns if c not in abx_cols] # 感染部位
data_filter = table14_clear[base_cols + final_cols]

feature_cols = list(set(data_filter.columns) - set(abx_cols))
X = data_filter[feature_cols]
y = data_filter[final_cols]

X.shape, X.sum(), y.shape, y.sum()

((27968, 7),
 INFECTIONSITE2             4505
 INFECTIONSITE1             5879
 INFECTIONSITE3             3066
 INFECTIONSITE9              598
 OTHERINFECTIONSITE_flag     495
 INFECTIONSITE4              270
 INFECTIONSITE5             1480
 dtype: int64,
 (27968, 78),
 Acyclovir                               157.0
 Amikacin                                  8.0
 Amoxicillin                             388.0
 Amoxicillin/Clavulanic acid            5515.0
 Amphotericin B                            1.0
                                         ...  
 abacavir/lamivudine/dolutegravir          4.0
 ifampin/Isoniazid                         1.0
 tenofovir/emtricitabine                   5.0
 tenofovir/emtricitabine/bictegravir      80.0
 tenofovir/emtricitabine/rilpivirine       1.0
 Length: 78, dtype: float64)

In [34]:
abx14 = y

result = {}
for i in abx14.columns:
    summ = abx14[i].sum()
    if summ >= 0:
       result[i] = summ

for key, value in sorted(result.items(), key=lambda x: x[1], reverse=True):
    print(f'{key}: {value}')
    
print(len(result))

Amoxicillin/Clavulanic acid: 5515.0
Flomoxef: 5196.0
Cefazolin: 2371.0
Cefixime: 2166.0
Ciprofloxacin: 2071.0
Azithromycin: 2053.0
Cefuroxime: 1691.0
Piperacillin/Tazobactam: 1511.0
Cefoperazone/sulbactam: 1412.0
Peramivir: 1119.0
Baloxavir marboxil: 1085.0
Metronidazole: 1020.0
Cefadroxil: 918.0
Oseltamivir: 835.0
Levofloxacin: 720.0
Clindamycin: 624.0
Gentamicin: 623.0
Ceftriaxone: 610.0
Cephalexin: 517.0
Ampicillin: 423.0
Amoxicillin: 388.0
Doxycycline: 358.0
Nemonoxacin: 169.0
Acyclovir: 157.0
Valaciclovir: 129.0
Tenofovir alafenamide: 111.0
Cefotaxime: 102.0
Ceftazidime: 100.0
tenofovir/emtricitabine/bictegravir: 80.0
Cefepime: 79.0
Sulfamethoxazole/Trimethoprim: 66.0
Fosfomycin: 60.0
Vancomycin: 59.0
Moxifloxacin: 55.0
Famciclovir: 49.0
Nystatin: 33.0
Clarithromycin: 26.0
Pipemidic acid: 23.0
Oxacillin: 21.0
Ampicillin/Sulbactam: 20.0
Penicillin: 17.0
Cefoxitin: 15.0
Fluconazole: 15.0
Ceftizoxime: 14.0
Fenticonazole: 14.0
Meropenem: 13.0
Minocycline: 12.0
Amikacin: 8.0
Dicloxacil

In [35]:
col_sum = abx14.sum()

abx14_filter = abx14.loc[:, col_sum >= 0]

In [36]:
abx14_filter.columns

Index(['Acyclovir', 'Amikacin', 'Amoxicillin', 'Amoxicillin/Clavulanic acid',
       'Amphotericin B', 'Amphotericin B/liposome', 'Ampicillin',
       'Ampicillin/Sulbactam', 'Azithromycin', 'Baloxavir marboxil',
       'Cefadroxil', 'Cefazolin', 'Cefepime', 'Cefixime',
       'Cefoperazone/sulbactam', 'Cefotaxime', 'Cefoxitin', 'Ceftazidime',
       'Ceftazidime/Avibactam', 'Ceftizoxime', 'Ceftriaxone', 'Cefuroxime',
       'Cephalexin', 'Ciprofloxacin', 'Clarithromycin', 'Clindamycin',
       'Dicloxacillin', 'Doripenem', 'Doxycycline', 'Ertapenem',
       'Erythromycin', 'Ethambutol', 'Famciclovir', 'Fenticonazole',
       'Flomoxef', 'Fluconazole', 'Flucytosine', 'Fosfomycin', 'Ganciclovir',
       'Gentamicin', 'Griseofulvin', 'Imipenem/Cilastatin', 'Isoniazid',
       'Itraconazole', 'Lamivudine/Dolutegravir', 'Levofloxacin', 'Linezolid',
       'Meropenem', 'Metronidazole', 'Micafungin', 'Minocycline',
       'Moxifloxacin', 'Nemonoxacin', 'Nystatin', 'Oseltamivir', 'Oxacillin',

In [37]:
mask = abx14_filter.sum(axis=1) > 0
abx14_final = abx14_filter[mask].reset_index()

In [38]:
abx14_final

,index,Acyclovir,Amikacin,Amoxicillin,Amoxicillin/Clavulanic acid,Amphotericin B,Amphotericin B/liposome,Ampicillin,Ampicillin/Sulbactam,Azithromycin,...,Terbinafine,Tetracycline,Valaciclovir,Vancomycin,Zanamivir,abacavir/lamivudine/dolutegravir,ifampin/Isoniazid,tenofovir/emtricitabine,tenofovir/emtricitabine/bictegravir,tenofovir/emtricitabine/rilpivirine
0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
27963,27963,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27964,27964,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27965,27965,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
27966,27966,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
